# 5회차 · 모델 평가 & 전처리 — 정확도 너머를 보기

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/사용자명/ml-study-2weeks/blob/main/week2/05_evaluation/05_evaluation.ipynb)

**오늘의 목표 (약 2.5시간)**
- 정확도만 보면 안 되는 이유
- 혼동행렬(confusion matrix)
- 정밀도(precision) · 재현율(recall) · F1
- 교차검증(cross-validation)
- ⭐ 스케일링(정규화) 전/후 성능 비교 — 1·2회차 정규화가 실전에서 왜 필요한지

**진행 방식**: 개념 → 예제 실행 → 🔧 빈칸 채우기 순서입니다.

---
## 0. 정확도의 함정

암 환자가 100명 중 3명이라면, 무조건 "정상"이라고만 답해도 정확도 97%입니다. 하지만 정작 환자는 하나도 못 잡죠. 그래서 정확도 외의 지표가 필요합니다. 오늘은 유방암 진단 데이터로 실습해요.

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
import pandas as pd

data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target   # 0=악성, 1=양성

print("데이터 모양:", X.shape)
print("정답 분포:", pd.Series(y).value_counts().to_dict())

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

---
## 1. 혼동행렬 (Confusion Matrix)

모델이 무엇을 맞히고 무엇을 틀렸는지 4칸 표로 보여줍니다.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

model = LogisticRegression(max_iter=5000)
model.fit(X_train, y_train)
pred = model.predict(X_test)

cm = confusion_matrix(y_test, pred)
print(cm)

ConfusionMatrixDisplay(cm, display_labels=["malignant", "benign"]).plot()
plt.show()

> 💡 대각선(왼쪽위–오른쪽아래)이 맞힌 것, 나머지가 틀린 것입니다.

---
## 2. 정밀도 · 재현율 · F1

- **정밀도(precision)**: "양성이라 예측한 것 중 진짜 양성 비율"
- **재현율(recall)**: "실제 양성 중 모델이 잡아낸 비율" — 암 진단에서 특히 중요
- **F1**: 둘의 균형 점수

`classification_report` 로 한 번에 봅니다.

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_test, pred, target_names=["malignant", "benign"]))

### 🔧 과제 2-1
결정트리로 학습·예측한 뒤 `classification_report` 를 출력하세요.

In [ ]:
from sklearn.tree import DecisionTreeClassifier

tree = DecisionTreeClassifier(random_state=42)
tree.fit(X_train, y_train)
tree_pred = ___          # TODO: 예측

# TODO: classification_report 출력
print(___)

---
## 3. 교차검증 (Cross-Validation)

데이터를 한 번만 나누면 "운 좋게/나쁘게" 나뉠 수 있습니다. 교차검증은 여러 번 나눠 평균을 내 더 믿을 만한 점수를 줍니다.

In [ ]:
from sklearn.model_selection import cross_val_score

scores = cross_val_score(LogisticRegression(max_iter=5000), X, y, cv=5)
print("5번 점수:", [round(s, 3) for s in scores])
print("평균:", round(scores.mean(), 3))

---
## 4. ⭐ 스케일링 전/후 비교

특징들의 값 범위가 제각각이면(어떤 건 0~1, 어떤 건 0~2000) 모델이 헷갈립니다. 그래서 **스케일링(표준화)** 으로 범위를 맞춰줍니다 — 1·2회차에서 손으로 짰던 그 정규화/표준화예요. scikit-learn엔 `StandardScaler` 가 있습니다.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

# 스케일링 전
m1 = LogisticRegression(max_iter=5000)
m1.fit(X_train, y_train)
acc_before = accuracy_score(y_test, m1.predict(X_test))

# 스케일링 후
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)   # 학습 데이터로 기준 학습 + 변환
X_test_s = scaler.transform(X_test)         # 테스트는 같은 기준으로 변환만

m2 = LogisticRegression(max_iter=5000)
m2.fit(X_train_s, y_train)
acc_after = accuracy_score(y_test, m2.predict(X_test_s))

print(f"스케일링 전: {acc_before:.3f}")
print(f"스케일링 후: {acc_after:.3f}")

> 💡 `fit_transform` 은 학습 데이터에만, `transform` 은 테스트 데이터에. 테스트 데이터의 정보가 학습에 새어들면 안 되기 때문이에요(중요한 습관!).

### 🔧 오늘의 미니 과제 (제출용)
스케일링 전/후의 성능을 **표로 정리**하세요. 정확도뿐 아니라 재현율도 함께 비교합니다.

In [ ]:
from sklearn.metrics import recall_score
import pandas as pd

# 전
rec_before = recall_score(y_test, m1.predict(X_test))
# 후  TODO: m2 와 X_test_s 로 재현율 계산
rec_after = ___

result = pd.DataFrame({
    "구분": ["스케일링 전", "스케일링 후"],
    "정확도": [round(acc_before, 3), round(acc_after, 3)],
    "재현율": [round(rec_before, 3), round(rec_after, 3)],
})
result

---
## 마무리 & 제출

**제출 방법**
1. `런타임 → 모두 실행` 으로 전체를 돌려 결과를 채웁니다.
2. `파일 → GitHub에 사본 저장` → 경로를 `members/본인이름/05_evaluation.ipynb` 로 지정합니다.
3. 커밋 메시지: `5회차 완료`

**복습 팁**: 막혔던 셀 아래에 마크다운 셀을 추가해 "왜 헷갈렸는지"를 한두 줄 적어두면 최고의 복습 노트가 됩니다.


**다음 회차 예고 (6회차)**: 마지막! Keras로 작은 신경망을 만들어 손글씨 숫자(MNIST)를 분류하는 딥러닝을 직접 돌려봅니다.